In [ ]:
#workshopping Historical_FWI.py generalisation 

#first need to recreate the iberia work but more efficient to check data processing is working.



In [ ]:
#cylc stuff 

[task parameters]
    
    member = 1..15

[scheduling]

    [[graph]]
        P1Y = """
            hadgem_historical_fwi<member>
        """

[runtime]
    [[hadgem_historical_fwi<member>]]
        script = """
        set -eux
        conda activate sowf
        cd /data/users/bob.potts/StateOfFires_2025-26/code
        python hadgem_historical_fwi.py
        """

        platform = spice

        [[[directives]]]
            --mem = 100G
            --time = 60
            --cpus-per-task = 1
            --ntasks = 1

In [1]:
import iris
import numpy as np
from utils.constrain_cubes_standard import *
from utils.cubefuncs import *
import time
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='iris')

In [30]:
# ...existing code...
import numpy as np

file_a = "/data/scratch/chantelle.burton/SoW2526/output/ERA5_FWI_1960-2013_Iberia95%.dat"
file_b = "/data/scratch/bob.potts/sowf/test_output/IRIS_ERA5_FWI_1960-2013_Iberia95%.dat"

a = np.loadtxt(file_a)
b = np.loadtxt(file_b)

print("len a:", len(a), "len b:", len(b))
print("first 5 a:", a[:5])
print("first 5 b:", b[:5])
import numpy as np

print("len a:", len(a), "len b:", len(b))
n = min(len(a), len(b))
print("max abs diff:", np.max(np.abs(a[:n]-b[:n])))
print("mean abs diff:", np.mean(np.abs(a[:n]-b[:n])))


len a: 54 len b: 54
first 5 a: [29.70566406 46.59042969 46.44121094 40.10722656 39.90292969]
first 5 b: [29.70566406 46.59042969 46.44121094 40.10722656 39.90292969]
len a: 54 len b: 54
max abs diff: 0.0
mean abs diff: 0.0


In [31]:
import geopandas as gpd
gdf = gpd.read_file('/data/users/chantelle.burton/Attribution/StateOfFires_2025-26/SoW2526_Focal_MASTER_20260218.shp')
print(gdf.columns)
print(gdf['name'].unique()) 

Index(['UID', 'name', 'geometry'], dtype='str')
<StringArray>
[                      'Northwest Iberia',
     'Midwestern Canadian Shield forests',
 'Chilean Temperate Forests and Matorral',
                     'Scottish Highlands',
                  'Southeast South Korea']
Length: 5, dtype: str


In [ ]:

############# User inputs here #############
Country = 'Iberia'
START_YEAR = 1960
END_YEAR = 2013
CSV_EXPORT = True #True for CSV, False for .dat
# Options: 'South Korea' (3), 'Iberia' (8), 'Scotland' (7)
############# User inputs end here #############

folder = '/data/scratch/chantelle.burton/SoW2526/'
shp_file = '/data/users/chantelle.burton/Attribution/StateOfFires_2025-26/SoW2526_Focal_MASTER_20260218.shp'

# Set up the 2025 files and months automatically
if Country == 'South Korea':
    print('Running South Korea')
    Month = 3
    month = 'March'
    percentile = 95
    shape_name = 'South Korea'
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-01-01-2025-05-31_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

elif Country == 'Iberia':
    print('Running Iberia')
    Month = 8
    month = 'Aug'
    percentile = 95
    shape_name = 'Northwest Iberia'
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

elif Country == 'Scotland':
    print('Running Scotland')
    Month = 7
    month = 'July'
    percentile = 95
    shape_name = 'Scottish Highlands'
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

start_time = time.time()

# Make hist and histnat arrays, and print them because of memory limits
folder = '/data/scratch/chantelle.burton/SoW2526/Y2526FWI/'
index_filestem1 = 'historicalExt'
index_filestem2 = 'historicalNatExt'
index_name = 'canadian_fire_weather_index'


members = np.arange(1,106)
histarray = []
for member in members:
    print ('hist',member)
    for n in np.arange(1,6):
        try:
            if member < 10:
                hist = iris.load_cube(folder+'FWI_HadGEM3-A-N216_r00'+str(member)+'i1p'+str(n)+'_'+index_filestem1+'_20230601-20250201_global_day.nc', index_name)
            elif member > 9 and member < 100:
                hist = iris.load_cube(folder+'FWI_HadGEM3-A-N216_r0'+str(member)+'i1p'+str(n)+'_'+index_filestem1+'_20230601-20250201_global_day.nc', index_name)
            else:
                hist = iris.load_cube(folder+'FWI_HadGEM3-A-N216_r'+str(member)+'i1p'+str(n)+'_'+index_filestem1+'_20230601-20250201_global_day.nc', index_name)           
            hist = contrain_to_sow_shapefile(hist, '/data/users/chantelle.burton/Attribution/StateOfFires_2025-26/SoW2526_Focal_MASTER_20260218.shp', 'Northwest Iberia')
            hist = ConstrainToYear(hist) 
            iris.save(hist,'/data/scratch/chantelle.burton/SoW2526/output/HadGEMtest.nc')
            hist = CountryPercentile(hist, percentile)
            hist = TimePercentile(hist, percentile)
            hist = np.ravel(hist.data)
            f = open('/data/scratch/bob.potts/SoW2526/output/'+Country+'_UNCORRECTED_hist'+str(percentile)+'%.dat','a')
            np.savetxt(f,(hist),newline=',',fmt='%s')
            f.write('\n')
            f.close()
            histarray.append(hist)
        except IOError:
             pass 
     
histarray = np.array(histarray)
histarray = np.ravel(histarray)
print(repr(histarray)) 
exit()




# can do this in paralell to save time
histnatarray = []
members = np.arange(1,106)
for member in members:
    print ('histnat',member)
    for n in np.arange(1,6):
        try:
            if member < 10:
                histnat = iris.load_cube(folder+'FWI_HadGEM3-A-N216_r00'+str(member)+'i1p'+str(n)+'_'+index_filestem2+'_20230601-20250201_global_day.nc', index_name)
            elif member > 9 and member < 100:
                histnat = iris.load_cube(folder+'FWI_HadGEM3-A-N216_r0'+str(member)+'i1p'+str(n)+'_'+index_filestem2+'_20230601-20250201_global_day.nc', index_name)
            else:
                histnat = iris.load_cube(folder+'FWI_HadGEM3-A-N216_r'+str(member)+'i1p'+str(n)+'_'+index_filestem2+'_20230601-20250201_global_day.nc', index_name)           
            histnat = contrain_to_sow_shapefile(histnat, '/data/users/chantelle.burton/Attribution/StateOfFires_2025-26/SoW2526_Focal_MASTER_20260218.shp', 'Northwest Iberia')
            histnat = ConstrainToYear(histnat)  
            histnat = CountryPercentile(histnat, percentile)
            histnat = TimePercentile(histnat, percentile)
            histnat = np.ravel(histnat.data)
            f = open('/data/scratch/chantelle.burton/SoW2526/output/'+Country+'_UNCORRECTED_histnat'+str(percentile)+'%.dat','a')
            np.savetxt(f,(histnat),newline=',',fmt='  %s')
            f.write('\n')
            f.close()
            histnatarray.append(histnat)
        except IOError:
            pass 
        
histnatarray = np.array(histnatarray)
histnatarray = np.ravel(histnatarray)
print(repr(histnatarray))

exit()



In [12]:
import glob
import re

############# User inputs here #############
Country = 'Iberia'
YEAR = 2024
# Options: 'South Korea' (3), 'Iberia' (8), 'Scotland' (7)
############# User inputs end here #############

folder = '/data/scratch/chantelle.burton/SoW2526/'
shp_file = '/data/users/chantelle.burton/Attribution/StateOfFires_2025-26/SoW2526_Focal_MASTER_20260218.shp'
data_folder = '/data/scratch/chantelle.burton/SoW2526/Y2526FWI/'

# Set up the 2025 files and months automatically
if Country == 'South Korea':
    print('Running South Korea')
    Month = 3
    month = 'March'
    percentile = 95
    shape_name = 'South Korea'
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-01-01-2025-05-31_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

elif Country == 'Iberia':
    print('Running Iberia')
    Month = 8
    month = 'Aug'
    percentile = 95
    shape_name = 'Northwest Iberia'
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

elif Country == 'Scotland':
    print('Running Scotland')
    Month = 7
    month = 'July'
    percentile = 95
    shape_name = 'Scottish Highlands'
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

start_time = time.time()
index_name = 'canadian_fire_weather_index'

# Process historicalExt
print("\n=== Processing historicalExt ===")
hist_files = sorted(glob.glob(data_folder + 'FWI_HadGEM3-A-N216_r*_historicalExt_20230601-20250201_global_day.nc'))
print(f"Found {len(hist_files)} historicalExt files")


histarray = []
for hist_file in hist_files:
    try:
        print(f"Loading {hist_file}")
        hist = iris.load_cube(hist_file, index_name)
        print(hist)
        hist = contrain_to_sow_shapefile(hist, shp_file, shape_name)
        hist = ConstrainToYear(hist)
        hist = CountryPercentile(hist, percentile)
        hist = TimePercentile(hist, percentile)
        hist = np.ravel(hist.data)
        
        # #Write to disk
        # with open(f'/data/scratch/chantelle.burton/SoW2526/output/{Country}_UNCORRECTED_hist{percentile}%.dat', 'a') as f:
        #     np.savetxt(f, hist, newline=',', fmt='%s')
        #     f.write('\n')
        
        histarray.append(hist)
    except IOError as e:
        print(f"Error loading {hist_file}: {e}")

histarray = np.array(histarray)
histarray = np.ravel(histarray)
print(f"histarray shape: {histarray.shape}")
print(repr(histarray))

# Process historicalNatExt
# print("\n=== Processing historicalNatExt ===")
# histnat_files = sorted(glob.glob(data_folder + 'FWI_HadGEM3-A-N216_r*_historicalNatExt_20230601-20250201_global_day.nc'))
# print(f"Found {len(histnat_files)} historicalNatExt files")

# histnatarray = []

# for histnat_file in histnat_files:
#     try:
#         print(f"Loading {histnat_file}")
#         histnat = iris.load_cube(histnat_file, index_name)
#         print(histnat.coords)
#         histnat = contrain_to_sow_shapefile(histnat, shp_file, shape_name)
#         histnat = ConstrainToYear(histnat,YEAR)
#         histnat = CountryPercentile(histnat, percentile)
#         histnat = TimePercentile(histnat, percentile)
#         histnat = np.ravel(histnat.data)
        
# #         # Write to disk
# #         with open(f'/data/scratch/chantelle.burton/SoW2526/output/{Country}_UNCORRECTED_histnat{percentile}%.dat', 'a') as f:
# #             np.savetxt(f, histnat, newline=',', fmt='%s')
# #             f.write('\n')
        
#         histnatarray.append(histnat)
#     except IOError as e:
#         print(f"Error loading {histnat_file}: {e}")

# histnatarray = np.array(histnatarray)
# histnatarray = np.ravel(histnatarray)
# print(f"histnatarray shape: {histnatarray.shape}")
# print(repr(histnatarray))

print(f"--- {np.round(time.time() - start_time, 2)} seconds ---")

Running Iberia

=== Processing historicalExt ===
Found 525 historicalExt files
Loading /data/scratch/chantelle.burton/SoW2526/Y2526FWI/FWI_HadGEM3-A-N216_r001i1p1_historicalExt_20230601-20250201_global_day.nc
Canadian Fire Weather Index / (1)   (time: 600; latitude: 324; longitude: 432)
    Dimension coordinates:
        time                             x              -               -
        latitude                         -              x               -
        longitude                        -              -               x
    Cell methods:
        0                           time: maximum (interval: 1 hour)
    Attributes:
        Conventions                 'CF-1.7'
        STASH                       m01s03i236
        least_significant_digit     np.int64(2)
        physics_version             np.int64(1)
        realization                 np.int64(1)
        source                      'Data from Met Office Unified Model'
        um_version                  '10.2'
        

TypeError: ConstrainToYear() missing 1 required positional argument: 'year'

In [ ]:
import glob
import re

############# User inputs here #############
Country = 'Iberia'
YEAR = 2024
# Options: 'South Korea' (3), 'Iberia' (8), 'Scotland' (7)
############# User inputs end here #############

folder = '/data/scratch/chantelle.burton/SoW2526/'
shp_file = '/data/users/chantelle.burton/Attribution/StateOfFires_2025-26/SoW2526_Focal_MASTER_20260218.shp'
data_folder = '/data/scratch/chantelle.burton/SoW2526/Y2526FWI/'

# Set up the 2025 files and months automatically
if Country == 'South Korea':
    print('Running South Korea')
    Month = 3
    month = 'March'
    percentile = 95
    shape_name = 'South Korea'
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-01-01-2025-05-31_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

elif Country == 'Iberia':
    print('Running Iberia')
    Month = 8
    month = 'Aug'
    percentile = 95
    shape_name = 'Northwest Iberia'
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

elif Country == 'Scotland':
    print('Running Scotland')
    Month = 7
    month = 'July'
    percentile = 95
    shape_name = 'Scottish Highlands'
    ERA5_2025 = iris.load_cube(folder+'Y2526FWI/FWI_ERA5_std_reanalysis_2025-06-01-2025-10-01_global_day_initialise-from=previous-and-use-numpy=False-and-code-src=copernicus-and-save-input-data=True.nc', 'canadian_fire_weather_index')

start_time = time.time()
index_name = 'canadian_fire_weather_index'

print("\n=== Processing historicalExt ===")
hist_files = sorted(glob.glob(data_folder + 'FWI_HadGEM3-A-N216_r*_historicalExt_20230601-20250201_global_day.nc'))
print(f"Found {len(hist_files)} historicalExt files")

# Regex to extract member and realization from filename
pattern = r'_r(\d+)i1p(\d+)_'

# Step 1: Load all files and add coordinates BEFORE merging
print("\n=== Loading all cubes ===")
hist_cubes = iris.cube.CubeList()

for hist_file in hist_files:
    try:
        # Extract member and realization from filename
        match = re.search(pattern, hist_file)
        if not match:
            print(f"Warning: Could not extract member/realization from {hist_file}")
            continue
        
        member = int(match.group(1))
        realization = int(match.group(2))
        
        print(f"Loading Member: {member}, Realization: {realization})")
        hist = iris.load_cube(hist_file, index_name)
        
        # Add member and realization as scalar coordinates BEFORE merging
        hist.add_aux_coord(iris.coords.AuxCoord(member, long_name='ensemble_member', units='1'))
        hist.add_aux_coord(iris.coords.AuxCoord(realization, long_name='realization', units='1'))
        
        hist_cubes.append(hist)
        
    except IOError as e:
        print(f"Error loading {hist_file}: {e}")

print(f"Loaded {len(hist_cubes)} cubes")    
print(hist_cubes[0])
# Step 2: Merge all cubes along member/realization dimensions
print("\n=== Merging cubes ===")
iris.util.equalise_attributes(hist_cubes)

hist_merged = hist_cubes.merge_cube()
#this is approx 350gb of data
print(f"Merged cube shape: {hist_merged.shape}")
print(hist_merged)

# Step 3: Apply operations to the merged cube (single operation on 525 ensemble members)
print("\n=== Applying constraints and percentiles ===")
hist_merged = contrain_to_sow_shapefile(hist_merged, shp_file, shape_name)
hist_merged = ConstrainToYear(hist_merged, YEAR)
hist_merged = CountryPercentile(hist_merged, percentile)
hist_merged = TimePercentile(hist_merged, percentile)

print(f"Processed cube shape: {hist_merged.shape}")
print(hist_merged)

# Step 4: Save and extract
print("\n=== Saving merged cube ===")
output_file = f'/data/scratch/bob.potts/sowf/test_output/{Country}_Uncorrected_hist_EXT{percentile}%.nc'


print(f"Saved to: {output_file}")


Running Iberia

=== Processing historicalExt ===
Found 525 historicalExt files

=== Loading all cubes ===
Loading Member: 1, Realization: 1)
Loading Member: 1, Realization: 2)
Loading Member: 1, Realization: 3)
Loading Member: 1, Realization: 4)
Loading Member: 1, Realization: 5)
Loading Member: 2, Realization: 1)
Loading Member: 2, Realization: 2)
Loading Member: 2, Realization: 3)
Loading Member: 2, Realization: 4)
Loading Member: 2, Realization: 5)
Loading Member: 3, Realization: 1)
Loading Member: 3, Realization: 2)
Loading Member: 3, Realization: 3)
Loading Member: 3, Realization: 4)
Loading Member: 3, Realization: 5)
Loading Member: 4, Realization: 1)
Loading Member: 4, Realization: 2)
Loading Member: 4, Realization: 3)
Loading Member: 4, Realization: 4)
Loading Member: 4, Realization: 5)
Loading Member: 5, Realization: 1)
Loading Member: 5, Realization: 2)
Loading Member: 5, Realization: 3)
Loading Member: 5, Realization: 4)
Loading Member: 5, Realization: 5)
Loading Member: 6, 

/home/users/bob.potts/.conda/envs/sowf/lib/python3.12/site-packages/iris/common/mixin.py:206: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microseconds = True`.
  warnings.warn(message, category=FutureWarning)


Processed cube shape: (105, 5)
Canadian Fire Weather Index / (1)          (ensemble_member: 105; realization: 5)
    Dimension coordinates:
        ensemble_member                                    x                 -
        realization                                        -                 x
    Scalar coordinates:
        latitude                           41.66668 degrees, bound=(39.722237, 43.611122) degrees
        longitude                          -6.66668701171875 degrees, bound=(-8.750030517578125, -4.583343505859375) degrees
        percentile_over_longitude_latitude 95 percent
        percentile_over_time               95 percent
        time                               2024-07-01 00:00:00, bound=(2024-01-01 00:00:00, 2025-01-01 00:00:00)
    Cell methods:
        0                                  time: maximum (interval: 1 hour)
    Attributes:
        Conventions                        'CF-1.7'
        STASH                              m01s03i236
        least_sign

PermissionError: [Errno 13] Permission denied: '/data/scratch/chantelle.burton/SoW2526/output/Iberia_HIST_merged_95%.nc'